# Projet de Prédiction du S&P 500 avec Machine Learning
Ce notebook contient l'implémentation complète d'un modèle de Forêt Aléatoire (Random Forest) pour prédire les mouvements du S&P 500.

In [ ]:
pip install yfinance

In [ ]:
# Cellule 1 : Importation et récupération des données
import yfinance as yf    # Pour télécharger les données boursières de Yahoo Finance
import pandas as pd      # Pour la manipulation des données sous forme de tableaux (DataFrames)
import os                # Pour vérifier la présence de fichiers sur le disque dur

# Stratégie d'évitement du téléchargement répété :
# Si le fichier CSV existe déjà localement, on le charge directement pour gagner du temps
if os.path.exists("sp500.csv"):
    sp500 = pd.read_csv("sp500.csv", index_col=0)
# Sinon, on télécharge tout l'historique disponible du S&P 500 depuis Yahoo Finance et on le sauvegarde
else:
    sp500 = yf.Ticker("^GSPC")
    sp500 = sp500.history(period="max")
    sp500.to_csv("sp500.csv")

# On s'assure que l'index du tableau (les dates) soit correctement converti au format Datetime de Pandas
sp500.index = pd.to_datetime(sp500.index)

# Affichage des premières et dernières lignes du tableau
sp500

In [ ]:
# Cellule 2 : Visualisation de base
# Tracé d'un graphique linéaire historique basé sur la colonne des prix de clôture ("Close")
sp500.plot.line(y="Close", use_index=True)

In [ ]:
# Cellule 3 : Nettoyage et création des cibles (Target)
# Suppression des colonnes inutiles pour un indice boursier global (pas de dividendes ou de fractionnements d'actions directs)
del sp500["Dividends"]
del sp500["Stock Splits"]

# Création de la colonne "Tomorrow" : on décale le prix de clôture d'un jour vers le haut pour avoir le prix du lendemain sur la ligne d'aujourd'hui
sp500["Tomorrow"] = sp500["Close"].shift(-1)

# Création de la colonne "Target" (la cible) :
# Si le prix de demain est supérieur à celui d'aujourd'hui, le résultat est True (converti en 1), sinon False (converti en 0)
sp500["Target"] = (sp500["Tomorrow"] > sp500["Close"]).astype(int)

# On filtre les données pour ne garder que les dates à partir du 1er janvier 1990 (le marché d'avant étant trop daté)
sp500 = sp500.loc["1990-01-01":].copy()

In [ ]:
# Cellule 4 : Premier entraînement du modèle
# Importation de l'algorithme des Forêts Aléatoires (Random Forest)
from sklearn.ensemble import RandomForestClassifier

# Initialisation du modèle : 100 arbres de décision, au moins 100 échantillons pour diviser un nœud (évite le surapprentissage), et un état aléatoire fixe
model = RandomForestClassifier(n_estimators=100, min_samples_split=100, random_state=1)

# Division chronologique des données :
train = sp500.iloc[:-100]  # Tout sauf les 100 derniers jours pour l'entraînement
test = sp500.iloc[-100:]   # Les 100 derniers jours réservés pour le test

# Définition des variables prédictives (les données de la journée en cours)
predictors = ["Close", "Volume", "Open", "High", "Low"]

# Entraînement du modèle sur les données d'entraînement pour faire le lien entre les variables et la cible (Target)
model.fit(train[predictors], train["Target"])

In [ ]:
# Cellule 5 : Évaluation du premier modèle
# Importation de la métrique de score de précision (mesure la fiabilité des signaux d'achat)
from sklearn.metrics import precision_score

# Génération des prédictions (0 ou 1) sur le jeu de test
preds = model.predict(test[predictors])

# Conversion des prédictions en Série Pandas pour pouvoir l'aligner avec les dates du jeu de test
preds = pd.Series(preds, index=test.index)

# Calcul du score de précision : sur toutes les fois où le modèle a prédit une hausse (1), combien de fois a-t-il eu raison ?
precision_score(test["Target"], preds)

In [ ]:
# Cellule 6 : Graphique des prédictions
# Fusion de la réalité (Target) et des prédictions (preds) côte à côte dans un même tableau pour affichage
combined = pd.concat([test["Target"], preds], axis=1)

# Affichage du graphique comparatif entre les mouvements réels et les choix du modèle
combined.plot()

In [ ]:
# Cellule 7 : Fonctions de Backtesting (Validation historique globale)
# Fonction qui entraîne le modèle sur un jeu d'entraînement, fait des prédictions sur un jeu de test et renvoie le tableau combiné
def predict(train, test, predictors, model):
    model.fit(train[predictors], train["Target"])
    preds = model.predict(test[predictors])
    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["Target"], preds], axis=1)
    return combined

# Fonction de simulation historique (Backtest) :
# start=2500 signifie qu'on attend d'avoir 10 ans de données historiques (2500 jours ouvrés) avant de faire la première prédiction.
# step=250 signifie qu'on avance et qu'on prédit année par année (250 jours de bourse par an).
def backtest(data, model, predictors, start=2500, step=250):
    all_predictions = []

    # Boucle qui parcourt le temps par tranches d'un an
    for i in range(start, data.shape[0], step):
        train = data.iloc[0:i].copy()            # Le modèle s'entraîne sur tout l'historique disponible jusqu'au jour "i"
        test = data.iloc[i:(i+step)].copy()      # Le modèle teste ses connaissances sur l'année suivante (jours "i" à "i+step")
        predictions = predict(train, test, predictors, model)
        all_predictions.append(predictions)      # Stockage des prédictions de l'année en cours

    # Fusion de toutes les prédictions annuelles en un seul grand tableau
    return pd.concat(all_predictions)

# Lancement du processus de backtest global sur l'historique du S&P 500
predictions = backtest(sp500, model, predictors)

In [ ]:
# Cellule 8 : Répartition des prédictions
# Compte le nombre de fois où le modèle a prédit une baisse (0) vs une hausse (1) sur tout l'historique du backtest
predictions["Predictions"].value_counts()

In [ ]:
# Cellule 9 : Score de précision du Backtest
# Calcul du score de précision final du modèle sur l'ensemble du backtest historique
precision_score(predictions["Target"], predictions["Predictions"])

In [ ]:
# Cellule 10 : Tendance naturelle du marché
# Calcul du pourcentage de jours de hausse réelle dans le marché (la tendance naturelle du marché à monter sans modèle)
predictions["Target"].value_counts() / predictions.shape[0]

In [ ]:
# Cellule 11 : Ingénierie des caractéristiques (Indicateurs Techniques)
# Définition de différentes fenêtres temporelles (2 jours, 1 semaine, 3 mois, 1 an, 4 ans)
horizons = [2, 5, 60, 250, 1000]
new_predictors = []

# Création de nouvelles variables contextuelles basées sur les tendances passées
for horizon in horizons:
    # Calcul de la moyenne mobile sur la période définie par l'horizon
    rolling_averages = sp500.rolling(horizon).mean()

    # 1. Ratio du prix actuel sur la moyenne mobile : indique si le cours est anormalement haut ou bas par rapport à sa tendance
    ratio_column = f"Close_Ratio_{horizon}"
    sp500[ratio_column] = sp500["Close"] / rolling_averages["Close"]

    # 2. Somme glissante des hausses passées : indique combien de jours de hausse ont eu lieu sur les X derniers jours
    trend_column = f"Trend_{horizon}"
    sp500[trend_column] = sp500.shift(1).rolling(horizon).sum()["Target"]

    # Ajout de ces nouveaux indicateurs à notre liste de variables prédictives
    new_predictors += [ratio_column, trend_column]

In [ ]:
# Cellule 12 : Nettoyage final des données
# La moyenne sur 1000 jours génère des valeurs manquantes (NaN) au début du tableau.
# On supprime toutes les lignes incomplètes (sauf pour la colonne "Tomorrow" qui peut légitimement être vide le tout dernier jour).
sp500 = sp500.dropna(subset=sp500.columns[sp500.columns != "Tomorrow"])

# Affichage du nouveau tableau enrichi de ses indicateurs techniques
sp500

In [ ]:
# Cellule 13 : Amélioration du modèle
# Instanciation d'un nouveau modèle plus robuste : 200 arbres et contrainte assouplie (min_samples_split=50) pour capter plus de subtilités
model = RandomForestClassifier(n_estimators=200, min_samples_split=50, random_state=1)

In [ ]:
# Cellule 14 : Redéfinition de la prédiction avec seuil de confiance strict
# Redéfinition de la fonction de prédiction pour y intégrer un filtre de probabilité
def predict(train, test, predictors, model):
    model.fit(train[predictors], train["Target"])

    # Au lieu de prédire directement 0 ou 1, on demande la probabilité que le marché monte (colonne index 1)
    preds = model.predict_proba(test[predictors])[:,1]

    # RÈGLE STRICTE : On n'achète (1) que si le modèle est sûr à 60% ou plus que le marché va monter.
    preds[preds >= .6] = 1
    preds[preds < .6] = 0

    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["Target"], preds], axis=1)
    return combined

In [ ]:
# Cellule 15 : Lancement du Backtest amélioré
# Lancement d'un nouveau backtest avec les nouveaux indicateurs techniques et la règle de décision des 60%
predictions = backtest(sp500, model, new_predictors)

In [ ]:
# Cellule 16 : Répartition des nouvelles prédictions
# Vérification de la distribution des nouvelles prédictions (on remarque que le modèle est devenu beaucoup plus sélectif)
predictions["Predictions"].value_counts()

In [ ]:
# Cellule 17 : Nouveau score de précision
# Calcul de la nouvelle précision (le score doit normalement être supérieur à celui de la cellule 9)
precision_score(predictions["Target"], predictions["Predictions"])

In [ ]:
# Cellule 18 : Tendance du marché filtré & Affichage final
# Comparaison avec la distribution réelle des hausses et baisses sur cette période de temps filtrée
print(predictions["Target"].value_counts() / predictions.shape[0])

# Affichage final du tableau récapitulant les cibles réelles face aux prédictions sécurisées du modèle
predictions